# Baseline check notebook -- minimal_notebook image

Simulates a normal user session: install a package via pip, run computational
code against it (using pandas), clean up the pip installs, then validate that
cleanup actually restored the environment to its pre-check state.

Run with: `jupyter nbconvert --to notebook --execute baseline_check.ipynb`.
A failed assertion in any cell will make nbconvert exit non-zero.

In [ ]:
import subprocess
import sys


def installed_packages():
    result = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        capture_output=True, text=True, check=True,
    )
    return set(result.stdout.splitlines())


def pip_install(*packages):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *packages],
        check=True,
    )


def pip_uninstall(*packages):
    if packages:
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "--quiet", "-y", *packages],
            check=True,
        )


def fresh_process_can_import(module_name):
    result = subprocess.run([sys.executable, "-c", f"import {module_name}"], capture_output=True)
    return result.returncode == 0


baseline_packages = installed_packages()
print(f"Baseline package count: {len(baseline_packages)}")

## Install packages

In [ ]:
pip_install("pandas")

after_install_packages = installed_packages()
newly_installed = sorted(after_install_packages - baseline_packages)
print("Newly installed packages:", newly_installed)

assert newly_installed, "Expected pip install to add at least one new package"

## Run computational code using pandas

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "region": ["north", "north", "south", "south", "east"],
    "sales": [120, 150, 90, 110, 200],
    "units": [10, 12, 8, 9, 15],
})

summary = df.groupby("region").agg(total_sales=("sales", "sum"), total_units=("units", "sum"))
summary["avg_price"] = summary["total_sales"] / summary["total_units"]

print(summary)

assert summary.loc["north", "total_sales"] == 270
assert summary.loc["south", "total_units"] == 17
assert round(summary.loc["east", "avg_price"], 4) == round(200 / 15, 4)
print("PASS: pandas computation produced expected results")

## Clean up the pip installs from earlier

In [ ]:
package_names = [pkg.split("==")[0] for pkg in newly_installed]
pip_uninstall(*package_names)
print("Uninstalled:", package_names)

## Validate cleanup was successful

Import checks run in a fresh subprocess rather than this kernel, since this
kernel already has `pandas` cached in `sys.modules` from the cell above.

In [ ]:
after_cleanup_packages = installed_packages()
leftover = after_cleanup_packages - baseline_packages
missing = baseline_packages - after_cleanup_packages
pandas_still_importable = fresh_process_can_import("pandas")

print("Leftover packages after cleanup:", sorted(leftover))
print("Packages missing that were present at baseline:", sorted(missing))
print("pandas importable in a fresh process after cleanup:", pandas_still_importable)

ok = not leftover and not missing and not pandas_still_importable
if ok:
    print("PASS: environment restored to baseline after cleanup")
else:
    raise AssertionError("FAIL: environment was not cleanly restored to baseline")